In [24]:
from dotenv import load_dotenv
from config import Config
from factory import AgentFactory
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MCPTool, PromptAgentDefinition
from azure.identity import ClientSecretCredential, AzureCliCredential
from openai import OpenAI
from pathlib import Path
from typing import Any
import asyncio
import os
import webbrowser
import json


config = Config()
RUNTIME_DEBUG_PATH = Path(".") / "runtime_response_debug.json"

In [8]:
project_client = AIProjectClient(
    endpoint=config.foundry_project_endpoint,
    credential=AzureCliCredential(),
    allow_preview=True
)

In [29]:
agent = project_client.agents.get(agent_name=config.foundry_agent_name)


In [30]:
if agent.name:
    agent_name = agent.name
else:
    raise Exception("Agent not found")


In [31]:
openai = project_client.get_openai_client(agent_name=agent_name)

In [ ]:
def run_response_stream(openai_client: OpenAI, input_payload, agent_name: str, previous_response_id: str = None):
    """Run a response stream and print MCP/OAuth/approval events."""
    stream_kwargs: dict[str, Any] = {
        "model": "",
        "input": input_payload,
        "extra_body": {
            "agent_reference": {
                "type": "agent_reference",
                "name": agent_name,
                "version": "1",
            }
        },
    }
    if previous_response_id:
        stream_kwargs["previous_response_id"] = previous_response_id

    with openai_client.responses.stream(**stream_kwargs) as stream:
        for _ in stream:
            pass
        response = stream.get_final_response()

    return response

In [21]:
def _serialize(obj: Any) -> dict[str, Any] | list[Any] | str:
    """Best-effort serializer for SDK/OpenAI response objects."""
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    if hasattr(obj, "as_dict"):
        return obj.as_dict()
    if isinstance(obj, (dict, list, str)):
        return obj
    try:
        return json.loads(json.dumps(obj, default=lambda x: getattr(x, "__dict__", str(x))))
    except TypeError:
        return str(obj)

In [22]:
def get_oauth_consent_links(response: Any) -> list[str]:
    """Extract OAuth consent links from response output items."""
    payload = _serialize(response)
    if not isinstance(payload, dict):
        return []

    links: list[str] = []
    for item in payload.get("output", []) or []:
        if not isinstance(item, dict):
            continue
        if item.get("type") != "oauth_consent_request":
            continue

        consent_link = item.get("consent_link") or item.get("consent_url")
        if isinstance(consent_link, str) and consent_link.strip():
            links.append(consent_link.strip())

    return links

In [ ]:
def get_mcp_approval_request_ids(response: Any) -> list[str]:
    """Extract MCP approval request ids from response.output items."""
    payload = _serialize(response)
    if not isinstance(payload, dict):
        return []

    ids: list[str] = []
    for item in payload.get("output", []) or []:
        if isinstance(item, dict) and item.get("type") == "mcp_approval_request" and item.get("id"):
            ids.append(str(item["id"]))
    return ids

In [25]:
def print_response_diagnostics(response: Any) -> None:
    """Print concise response info and persist full payload."""
    payload = _serialize(response)
    if not isinstance(payload, dict):
        print("Unexpected response shape; unable to print diagnostics.")
        print(payload)
        return

    print(f"Response id: {payload.get('id')}")
    print(f"Response status: {payload.get('status')}")
    print(f"Output text: {payload.get('output_text')}")

    RUNTIME_DEBUG_PATH.parent.mkdir(parents=True, exist_ok=True)
    RUNTIME_DEBUG_PATH.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"Saved debug payload: {RUNTIME_DEBUG_PATH}")

In [34]:
response = run_response_stream(openai_client=openai, 
                               input_payload="Give me all flights to Paris",
                               agent_name=agent_name)

In [35]:
print_response_diagnostics(response)

Response id: resp_0310c4e1a33b0f72006a18e20565f88197b67d1e811a4f0ce6
Response status: completed
Output text: None
Saved debug payload: runtime_response_debug.json


In [28]:
consent_links = get_oauth_consent_links(response)
if consent_links:
    print("\nOAuth consent is required before MCP tool execution can continue.")
    for index, link in enumerate(consent_links, start=1):
        print(f"Consent URL {index}: {link}")       
        webbrowser.open(link)
        print(f"Opened consent URL {index} in browser.")
    print("Complete consent, then rerun the script.")    


OAuth consent is required before MCP tool execution can continue.
Consent URL 1: https://logic-swedencentral-001.consent.azure-apihub.net/login?data=eyJMb2dpbklkIjoibG9naWMtYXBpcy1zd2VkZW5jZW50cmFsX2U5NmU1NDE3LTY5MTgtNDRiYS04NTdhLWYxODBhYWNmMTNmZC1mbGlnaHRzZXJ2ZXItbWNwX3Rva2VuIiwiU2Vzc2lvbklkIjoiIiwiTG9nUnVudGltZVBvbGljeUlkIjpudWxsLCJMb2dDb25uZWN0aW9uSWQiOiJmbGlnaHRzZXJ2ZXItbWNwLjMwNzc3OWRkLTJiYWItNDZhMS04MjZhLWYwNzNkMDM5YWY0OSIsIkxvZ0Nvbm5lY3RvcklkIjoiZTk2ZTU0MTctNjkxOC00NGJhLTg1N2EtZjE4MGFhY2YxM2ZkLWZsaWdodHNlcnZlci1tY3AiLCJMb2dFbnZpcm9ubWVudElkIjoiYWktZTk2ZTU0MTctNjkxOC00NGJhLTg1N2EtZjE4MGFhY2YxM2ZkIiwiTG9nQWNjb3VudE5hbWUiOiJsb2dpYy1hcGlzLXN3ZWRlbmNlbnRyYWwiLCJFeHBpcmF0aW9uVGltZSI6IjIwMjYtMDUtMjlUMDE6Mzg6NDIuMzQ0MDM3WiIsIkRhdGEiOiJiVjdYV2tiMExWaWp3QnpNbk43UmZUSmhnbWg0RjNLMUZvOTZONkNwYXdZUUFMKzErUndid0c5RlRHOUo0dVEzV3cwQmpNL212SnVPb21FM1JyL2lrOXczdnNSTHZTb0V2bHJoeVdYZWFaWWo1TXV4R2dkNnhwa1JxN1BzR2JLL2JGWjU3bjZCV09EQ25vR1NRQUJDSEN1K2NWT0JGVWtOclJFQjFoUnJidUp5UTZyY3BvNmsvd2hQcUZqY0RlMU

In [ ]:
# After completing consent in the browser, run this cell to continue
# It passes previous_response_id so the agent picks up where it left off
response = run_response_stream(
    openai_client=openai,
    input_payload="Give me all flights to Paris",
    agent_name=agent_name,
    previous_response_id=response.id
)
print_response_diagnostics(response)

In [ ]:
# Auto-approve MCP tool calls and get final result
approval_ids = get_mcp_approval_request_ids(response)
if approval_ids:
    print(f"Approving {len(approval_ids)} MCP request(s) and continuing run")
    approval_inputs = [
        {
            "type": "mcp_approval_response",
            "approval_request_id": aid,
            "approve": True,
        }
        for aid in approval_ids
    ]
    response = run_response_stream(
        openai_client=openai,
        input_payload=approval_inputs,
        agent_name=agent_name,
        previous_response_id=response.id
    )
    print_response_diagnostics(response)
else:
    print("No MCP approval requests found — check the response output above.")